# Stock Info — Live Quote Block & Trading Signs

Fetch the payload behind the header of a set.or.th quote page: the trading **sign**
(`SP`/`NC`/`NP`/`CB`/`XD`…), current price and OHLC, the best bid/offer, and reference data.

Endpoint: `GET /api/set/stock/{symbol}/info`

This is the only endpoint in settfex that carries `sign`, and the only one that serves it for
**every** security type — warrants, DWs and DRs included. The stock list has no sign field at
all, and index compositions cover common stocks only.

> There is deliberately **no** `lang` argument: `?lang=en` and `?lang=th` return byte-identical
> payloads, and both names always come back as `name_en` / `name_th`.

## Setup

In [ ]:
# !pip install settfex

import asyncio

from settfex.services.set import Stock, get_stock_info
from settfex.services.set.stock.info import parse_signs

## 1. A normal, untagged stock

`sign` is `""` for a security with nothing posted against it, so `signs` is an empty list.

In [ ]:
info = await get_stock_info("CPALL")

print(f"Name        : {info.name_en}")
print(f"Type        : {info.security_type} ({info.asset_type})")
print(f"Signs       : {info.signs or '(none)'}   suspended={info.is_suspended}")
print(f"Status      : {info.market_status}  @ {info.market_date_time}")
print(f"Price       : prior={info.prior} last={info.last} change={info.change}")
print(f"Best bid/ask: {info.best_bid} / {info.best_offer}")
print(f"52-week     : {info.low_52_weeks} - {info.high_52_weeks}")

## 2. A suspended stock

SET packs every active sign into **one comma-separated string**, so `sign == "SP"` misses most
suspended symbols. Use `signs`, `has_sign()` or `is_suspended` instead.

A halted symbol still returns HTTP 200 — with `last`/OHLC/volume all `None` and an empty book.
Use `prior` for its last known price.

In [ ]:
info = await get_stock_info("GRAND")

print(f"raw sign     : {info.sign!r}")
print(f"parsed signs : {info.signs}")
print(f"has_sign(sp) : {info.has_sign('sp')}   # case-insensitive")
print(f"is_suspended : {info.is_suspended}")
print(f"market_status: {info.market_status}")
print(f"last / prior : {info.last} / {info.prior}")
print(f"book         : bids={info.bids} offers={info.offers}")

### `is_suspended` reads the sign, not `market_status`

`market_status` says `'Closed'` for every symbol outside trading hours, so it cannot tell you
whether a particular symbol is suspended. Only the per-symbol sign can.

In [ ]:
for symbol in ["CPALL", "INGRS", "GRAND"]:
    i = await get_stock_info(symbol)
    print(
        f"{i.symbol:8s} status={i.market_status:8s} signs={str(i.signs):28s} suspended={i.is_suspended}"
    )

## 3. Screen a watchlist

Concurrent fetches — one request per symbol.

In [ ]:
WATCHLIST = ["CPALL", "PTT", "KBANK", "INGRS", "GRAND", "TTCL"]

infos = await asyncio.gather(*(get_stock_info(s) for s in WATCHLIST))

print(f"{'SYMBOL':10s} {'SIGNS':22s} {'LAST':>8s}  STATUS")
print("-" * 55)
for i in infos:
    flag = ", ".join(i.signs) or "-"
    last = f"{i.last:.2f}" if i.last is not None else "-"
    print(f"{i.symbol:10s} {flag:22s} {last:>8s}  {i.market_status}")

## 4. Every security type

The quote block covers stocks, warrants, DWs, DRs, ETFs, preferred and foreign lines.
Type-specific fields stay `None` where they do not apply.

In [ ]:
SAMPLES = {
    "CPALL": "common stock",
    "A5-W5": "warrant",
    "AAV13C2610A": "derivative warrant",
    "AAOI03": "depositary receipt",
    "1DIV": "ETF",
    "BH-P": "preferred",
}

for symbol, label in SAMPLES.items():
    i = await get_stock_info(symbol)
    print(f"{i.symbol:14s} {label:20s} type={i.security_type} ({i.asset_type})  last={i.last}")

### Warrant / DW terms

`exercise_ratio` is kept **verbatim** as SET's display string (`'0.36 : 1'`, `'3,000 : 1'`).

In [ ]:
dw = await get_stock_info("AAV13C2610A")

print(f"underlying   : {dw.underlying}")
print(f"exercise     : {dw.exercise_price} {dw.exercise_price_unit}")
print(f"ratio        : {dw.exercise_ratio!r}")
print(f"multiplier   : {dw.multiplier}")
print(f"maturity     : {dw.maturity_date:%Y-%m-%d}  (last trading {dw.last_trading_date:%Y-%m-%d})")
print(f"ttm          : {dw.ttm} days")
print(f"moneyness    : {dw.moneyness_status} {dw.moneyness_percent}%")

### ETF indicative NAV

`inav` is a nested block, populated for ETFs only.

In [ ]:
etf = await get_stock_info("1DIV")

print(f"underlying: {etf.underlying}")
print(f"last      : {etf.last}")
if etf.inav is not None:
    print(f"iNAV      : {etf.inav.inav} ({etf.inav.change:+}, {etf.inav.percent_change:+}%)")

## 5. The `Stock` class

`get_info()` is deliberately **not cached** — it is live trading state, unlike
`get_asset_type()` or `get_dr_profile()`.

In [ ]:
stock = Stock("ingrs")  # symbols are normalized to uppercase

print(f"symbol      : {stock.symbol}")
print(f"signs       : {await stock.get_signs()}")
print(f"is_suspended: {await stock.is_suspended()}")

## 6. Market-wide: every symbol carrying SP

One request per symbol would be ~4,000 requests. Index **compositions** carry the same `sign`
field for all 929 common stocks in ~36 requests.

Two traps:

1. SET's own `INDUSTRY`-level compositions come back **empty** — use `SECTOR` for SET, and the
   `-m` INDUSTRY indices for mai.
2. This route reaches common stocks (`securityType == 'S'`) only. A suspended **warrant, DW, DR,
   ETF or unit trust** is invisible to it — check those individually with `get_stock_info()`.

In [ ]:
from settfex.services.set.index.composition import IndexCompositionService
from settfex.services.set.index.list import IndexListService


async def symbols_with_sign(code: str = "SP") -> dict[str, str]:
    index_list = await IndexListService().fetch_index_list()
    targets = [
        ix for ix in index_list.indices if ix.level == "SECTOR" or ix.query_symbol.endswith("-m")
    ]
    service = IndexCompositionService()
    results = await asyncio.gather(
        *(service.fetch_composition(ix.query_symbol) for ix in targets),
        return_exceptions=True,
    )
    flagged: dict[str, str] = {}
    for result in results:
        if isinstance(result, BaseException):
            continue
        for c in result.composition.stock_infos:
            if code in parse_signs(c.sign):
                flagged[c.symbol] = c.sign or ""
    return dict(sorted(flagged.items()))


sp = await symbols_with_sign("SP")
print(f"{len(sp)} symbols carry SP\n")
for symbol, sign in sp.items():
    print(f"  {symbol:10s} {sign}")

### Cross-check one against the quote block

The composition `sign` and the quote-block `sign` are the same field from the same source.

In [ ]:
if sp:
    sample = next(iter(sp))
    i = await get_stock_info(sample)
    print(f"{sample}: composition={sp[sample]!r}  quote block={i.sign!r}  -> signs={i.signs}")

## 7. Error handling

| Situation | Behavior |
|---|---|
| Unknown symbol | HTTP 404 `{"message": "Invalid Stock Name"}` → `SymbolNotFoundError` |
| Empty symbol | `InvalidSymbolError`, before any request |
| Other HTTP failure | `FetchError` with `status_code` |

In [ ]:
from settfex.exceptions import InvalidSymbolError, SymbolNotFoundError

try:
    await get_stock_info("NOTAREALSYMBOL")
except SymbolNotFoundError as e:
    print(f"SymbolNotFoundError: {e}")

try:
    await get_stock_info("   ")
except InvalidSymbolError as e:
    print(f"InvalidSymbolError: {e}")

## 8. Serializing

`signs` and `is_suspended` are Pydantic **computed fields**, so the parsed form survives
`model_dump()` into Parquet or JSON — the audit trail does not have to be recomputed.

In [ ]:
info = await get_stock_info("GRAND")
dumped = info.model_dump(mode="json")

print("signs       :", dumped["signs"])
print("is_suspended:", dumped["is_suspended"])
print("fields      :", len(dumped))

---

**See also**

- [`01_stock_list.ipynb`](01_stock_list.ipynb) — the stock directory (no sign field)
- [`13_chart_quotation.ipynb`](13_chart_quotation.ipynb) — intraday series and latest traded price
- [`14_latest_historical_trading.ipynb`](14_latest_historical_trading.ipynb) — previous session's OHLCV
- [`15_market_index.ipynb`](15_market_index.ipynb) — index and sector compositions
- Docs: [`docs/settfex/services/set/info.md`](../../docs/settfex/services/set/info.md)